In [21]:
import torch
import json
from torch.utils.data import Dataset
from pathlib import Path

IMSITU_PATH = Path("/content/imsitu_dataset/")

with open(IMSITU_PATH / "imsitu_annotated.json", "r") as f:
    train_data = json.load(f)

train_data[: 5]

[{'verb': 'glaring',
  'gender': 'M',
  'agent_word': 'man',
  'image_path': 'glaring_215.jpg'},
 {'verb': 'talking',
  'gender': 'F',
  'agent_word': 'mother',
  'image_path': 'talking_90.jpg'},
 {'verb': 'patting',
  'gender': 'F',
  'agent_word': 'girl',
  'image_path': 'patting_236.jpg'},
 {'verb': 'begging',
  'gender': 'M',
  'agent_word': 'guy',
  'image_path': 'begging_143.jpg'},
 {'verb': 'rowing',
  'gender': 'M',
  'agent_word': 'man',
  'image_path': 'rowing_167.jpg'}]

In [ ]:
import json

with open(IMSITU_PATH / "imsitu_annotated_train.json", "r") as f:
    train_data = json.load(f)

with open(IMSITU_PATH / "imsitu_annotated_dev.json", "r") as f:
    val_data = json.load(f)

train_verbs = set([item['verb'] for item in train_data])
val_verbs = set([item['verb'] for item in val_data])

all_verbs = train_verbs.union(val_verbs)
verb_to_idx = {v: i for i, v in enumerate(sorted(list(all_verbs)))} 

# verb_to_idx

In [87]:
from PIL import Image

class ImsituDataset(Dataset):
    def __init__(self, source, transform=None, verb2idx=None):
        self.transform = transform

        with open(source, "r") as f:
            self.data = json.load(f)

        # We keep this for reference, but we don't rely on it for model dimension
        self.verbs = sorted(set(i['verb'] for i in self.data))
        self.verb_to_idx = verb2idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Load Image
        img_path = IMSITU_PATH / "of500_images_resized" / item['image_path']
        img = Image.open(img_path).convert("RGB")
        
        if self.transform:
            img = self.transform(img)
            
        # FIX 1: Get the verb string first, then map it to the index
        verb_str = item['verb']
        verb = self.verb_to_idx[verb_str] 
        
        # FIX 2: Convert "M"/"F" strings to 0/1 integers for future leakage steps
        # "M" -> 0, "F" -> 1
        gender_str = item['gender']
        gender = 0 if gender_str == "M" else 1

        return img, verb, gender

In [88]:
import torch.nn as nn
import torchvision.models as models

class BaselineResNet(nn.Module):
    def __init__(self, num_verbs):
        super(BaselineResNet, self).__init__()
        resnet = models.resnet50(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        feature_dim = resnet.fc.in_features
        self.classifier = nn.Linear(feature_dim, num_verbs)

    def forward(self, x):
        f = self.features(x)
        f = f.view(f.size(0), -1)
        logits = self.classifier(f)
        return logits

In [89]:
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from tqdm.notebook import tqdm

def train_baseline_model(train_loader, num_verbs, epochs=5):
    model = BaselineResNet(num_verbs).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    for epoch in tqdm(range(epochs), desc="Epochs"):
        model.train()
        total_loss = 0

        batch_iter = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)

        for images, verbs, _ in batch_iter:
            images, verbs = images.to(device), verbs.to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, verbs)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

    return model

In [90]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm.notebook import tqdm  # Specialized for Jupyter
import os

BATCH_SIZE = 32
LR = 1e-4
EPOCHS = 30
SAVE_PATH = "baseline_resnet_imsitu.pth"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device {device}")

Using device cuda


In [92]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = ImsituDataset(IMSITU_PATH / "imsitu_annotated_train.json", transform=train_transform, verb2idx=verb_to_idx)
val_dataset = ImsituDataset(IMSITU_PATH / "imsitu_annotated_dev.json", transform=val_transform, verb2idx=verb_to_idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Training on {len(train_dataset)} images. Validating on {len(val_dataset)} images.")

Training on 33603 images. Validating on 11211 images.


In [93]:
model = BaselineResNet(num_verbs=len(verb_to_idx)).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
best_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    
    for images, verbs, _ in progress_bar:
        images, verbs = images.to(DEVICE), verbs.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, verbs)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        progress_bar.set_postfix(loss=loss.item())

    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, verbs, _ in tqdm(val_loader, desc="Validating", leave=False):
            images, verbs = images.to(DEVICE), verbs.to(DEVICE)
            
            outputs = model(images)
            loss = criterion(outputs, verbs)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += verbs.size(0)
            correct += (predicted == verbs).sum().item()
            
    avg_val_loss = val_loss / len(val_loader)
    val_acc = 100 * correct / total
    
    scheduler.step(avg_val_loss)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | "
          f"Train Loss: {running_loss/len(train_loader):.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val Acc: {val_acc:.2f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"--> Best Model Saved (Acc: {best_acc:.2f}%)")

print("Training Complete.")

Epoch 1/30:   0%|          | 0/1051 [00:00<?, ?it/s]

Validating:   0%|          | 0/351 [00:00<?, ?it/s]

Epoch [1/30] | Train Loss: 4.7581 | Val Loss: 4.2059 | Val Acc: 15.79%
--> Best Model Saved (Acc: 15.79%)


Epoch 2/30:   0%|          | 0/1051 [00:00<?, ?it/s]

Validating:   0%|          | 0/351 [00:00<?, ?it/s]

...

Epoch 10/30:   0%|          | 0/1051 [00:00<?, ?it/s]

Validating:   0%|          | 0/351 [00:00<?, ?it/s]

Epoch [10/30] | Train Loss: 1.3334 | Val Loss: 3.9351 | Val Acc: 25.90%


Epoch 11/30:   0%|          | 0/1051 [00:00<?, ?it/s]

: 

## Calculating Leakage

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tqdm.notebook import tqdm

class Attacker(nn.Module):
    def __init__(self, input_dim, hidden_dim=300):
        super(Attacker, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(),
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(),
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.LeakyReLU(),
            
            nn.Linear(hidden_dim, 2)  # Binary Gender (0 or 1)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
def get_dataset_features(loader, num_verbs, device):
    """
    Extracts Ground Truth features (One-Hot Verbs) and Gender labels.
    """
    all_features = []
    all_genders = []
    
    for _, verbs, genders in tqdm(loader, desc="Extracting Dataset Features", leave=False):
        # Convert integer verbs to One-Hot float tensors
        one_hot = F.one_hot(verbs, num_classes=num_verbs).float()
        
        all_features.append(one_hot)
        all_genders.append(genders)
        
    return torch.cat(all_features).to(device), torch.cat(all_genders).to(device)

def get_model_features(baseline_model, loader, device):
    """
    Extracts Model Logits (features) and Gender labels.
    """
    baseline_model.eval()
    all_logits = []
    all_genders = []
    
    with torch.no_grad():
        for images, _, genders in tqdm(loader, desc="Extracting Model Logits", leave=False):
            images = images.to(device)
            
            # Forward pass to get logits (before Softmax)
            logits = baseline_model(images)
            
            all_logits.append(logits)
            all_genders.append(genders)
            
    return torch.cat(all_logits).to(device), torch.cat(all_genders).to(device)

In [ ]:
from tqdm import tqdm

def train_attacker(X_train, y_train, X_test, y_test, device, name="Attacker"):
    # Create datasets
    train_ds = TensorDataset(X_train, y_train)
    test_ds = TensorDataset(X_test, y_test)
    
    train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
    test_dl = DataLoader(test_ds, batch_size=128, shuffle=False)
    
    # Initialize attacker model
    input_dim = X_train.shape[1]
    model = Attacker(input_dim=input_dim).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=1e-4)  # Standard LR for MLP
    criterion = nn.CrossEntropyLoss()
    
    best_acc = 0.0
    epochs = 20  # MLPs converge quickly
    
    # ---- Epoch loop with tqdm ----
    for epoch in tqdm(range(epochs), desc=f"{name} Training"):
        model.train()

        # ---- Batch loop with tqdm ----
        train_iter = tqdm(train_dl, desc=f"Epoch {epoch+1}", leave=False)

        for features, targets in train_iter:
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
        
        # ---- Evaluation ----
        model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for features, targets in test_dl:
                outputs = model(features)
                _, predicted = torch.max(outputs.data, 1)
                total += targets.size(0)
                correct += (predicted == targets).sum().item()
        
        acc = 100 * correct / total
        if acc > best_acc:
            best_acc = acc
    
    print(f"{name} Results | Best Accuracy: {best_acc:.2f}%")
    return best_acc

In [ ]:
num_verbs = len(verb_to_idx)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. CALCULATE DATASET LEAKAGE (Lambda_D) ---
print("\n--- Calculating Dataset Leakage (Lambda_D) ---")
# Extract One-Hot Vectors
d_X_train, d_y_train = get_dataset_features(train_loader, num_verbs, DEVICE)
d_X_val, d_y_val = get_dataset_features(val_loader, num_verbs, DEVICE)

# Train Attacker on Ground Truth
lambda_d = train_attacker(d_X_train, d_y_train, d_X_val, d_y_val, DEVICE, name="Dataset Leakage")


--- Calculating Dataset Leakage (Lambda_D) ---


Dataset Leakage Training: 100%|██████████| 20/20 [00:42<00:00,  2.14s/it]       

Dataset Leakage Results | Best Accuracy: 68.18%


In [ ]:
num_verbs = len(verb_to_idx)

# A. Instantiate the empty architecture
loaded_model = BaselineResNet(num_verbs=num_verbs).to(DEVICE)

# B. Load the weights
loaded_model.load_state_dict(torch.load("baseline_resnet_imsitu.pth", map_location=DEVICE))

# C. Set to Evaluation Mode (Critical for consistent feature extraction)
loaded_model.eval()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


BaselineResNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          (0): Con

In [ ]:
print("\n--- Calculating Model Leakage (Lambda_M) ---")
# Extract Model Logits
m_X_train, m_y_train = get_model_features(loaded_model, train_loader, DEVICE)
m_X_val, m_y_val = get_model_features(loaded_model, val_loader, DEVICE)

# Train Attacker on Model Predictions
lambda_m = train_attacker(m_X_train, m_y_train, m_X_val, m_y_val, DEVICE, name="Model Leakage")


--- Calculating Model Leakage (Lambda_M) ---


Model Leakage Training: 100%|██████████| 20/20 [00:42<00:00,  2.14s/it]     

Model Leakage Results | Best Accuracy: 73.78%


In [ ]:
amplification = lambda_m - lambda_d

print("\n" + "="*40)
print(f"Dataset Leakage (Lambda_D): {lambda_d:.2f}%")
print(f"Model Leakage   (Lambda_M): {lambda_m:.2f}%")
print(f"Bias Amplification (Delta): {amplification:.2f}%")
print("="*40)


Dataset Leakage (Lambda_D): 68.18%
Model Leakage   (Lambda_M): 73.78%
Bias Amplification (Delta): 5.60%
